# 08 -- HAC Standard Errors and Weighted Least Squares: polars_reg vs R

This notebook verifies that `polars_reg` produces results equivalent to R for:

- **HAC standard errors**: Newey-West (NW) and Driscoll-Kraay (DK)
- **Weighted Least Squares (WLS)**: analytic weights with iid and robust SEs

**Datasets**:
- `Produc` from R's `plm` package (816 obs = 48 states x 17 years) for HAC tests
- `mtcars` from base R (32 obs) for WLS tests

**R packages used**: `sandwich`, `lmtest`, `fixest`

In [ ]:
import sys, os
import numpy as np
import polars as pl

# Ensure polars_reg is importable
sys.path.insert(0, os.path.abspath("../.."))
import polars_reg as pr

# Verification helpers
sys.path.insert(0, os.path.abspath("."))
from r_helper import load_r_dataset, run_r_regression, compare, compare_scalar, R_EXTRACT

## Load datasets

In [ ]:
# Produc: 48 US states x 17 years (1970-1986)
df_produc = load_r_dataset("Produc", package="plm")
print(f"Produc shape: {df_produc.shape}")
print(df_produc.head(3))

In [ ]:
# Create log-transformed columns
df_produc = df_produc.with_columns([
    pl.col("gsp").log().alias("lgsp"),
    pl.col("pcap").log().alias("lpcap"),
    pl.col("pc").log().alias("lpc"),
    pl.col("emp").log().alias("lemp"),
])

print(df_produc.select(["state", "year", "lgsp", "lpcap", "lpc", "lemp", "unemp"]).head(5))

In [ ]:
# mtcars: 32 observations
df_mtcars = load_r_dataset("mtcars")
print(f"mtcars shape: {df_mtcars.shape}")
print(df_mtcars.head(3))

In [ ]:
# Create weight column for WLS: w = 1/wt
df_mtcars = df_mtcars.with_columns(
    (1.0 / pl.col("wt")).alias("w")
)

print(df_mtcars.select(["mpg", "wt", "hp", "w"]).head(5))

In [ ]:
# Save CSVs so R scripts read identical data
csv_produc = os.path.abspath("_produc_hac.csv")
csv_mtcars = os.path.abspath("_mtcars_wls.csv")

df_produc.write_csv(csv_produc)
df_mtcars.write_csv(csv_mtcars)

print(f"Produc saved to {csv_produc}")
print(f"mtcars saved to {csv_mtcars}")

In [ ]:
# Common R preamble for Produc
R_PRODUC = f'''
df <- read.csv("{csv_produc}")
df$lgsp <- log(df$gsp)
df$lpcap <- log(df$pcap)
df$lpc <- log(df$pc)
df$lemp <- log(df$emp)
'''

# Common R preamble for mtcars
R_MTCARS = f'''
df <- read.csv("{csv_mtcars}")
df$w <- 1.0 / df$wt
'''

---
## HAC Standard Errors

### Test 1: Newey-West SEs (time-series HAC)

Standard Newey-West on pooled panel data. `polars_reg` sorts by time and applies
the Bartlett kernel with bandwidth = `floor(4*(N/100)^(2/9))`.

R comparison uses `sandwich::NeweyWest()` with the same bandwidth and `prewhite=FALSE`.

In [ ]:
# polars_reg: Newey-West SEs
res_nw = pr.ols(
    "lgsp ~ lpcap + lpc + lemp + unemp",
    data=df_produc,
    vcov="NW",
    time="year",
)
res_nw.summary()

In [ ]:
# Compute bandwidth to pass to R (must match polars_reg default)
n_produc = df_produc.height
bw_nw = int(np.floor(4 * (n_produc / 100) ** (2 / 9)))
print(f"N = {n_produc}, NW bandwidth = {bw_nw}")

In [ ]:
# R: sandwich::NeweyWest with matching bandwidth
r_nw = run_r_regression(f'''
library(sandwich)
library(lmtest)
{R_PRODUC}

# Sort by time (matching polars_reg behavior)
df <- df[order(df$year), ]

model <- lm(lgsp ~ lpcap + lpc + lemp + unemp, data=df)
vcov_mat <- NeweyWest(model, lag={bw_nw}, prewhite=FALSE, adjust=TRUE)
{R_EXTRACT}
''')
print("R coefficients:", r_nw.coef)
print("R SEs:", r_nw.se)

In [ ]:
print("=== Newey-West SEs ===")
compare(res_nw, r_nw, rtol=5e-3, label="NW")

---
### Test 2: Driscoll-Kraay SEs

DK aggregates score vectors by time period, then applies Newey-West on the
aggregated T x k matrix. Bandwidth = `floor(4*(T/100)^(2/9))`.

R comparison uses `fixest::feols()` with `vcov="DK"`.

In [ ]:
# polars_reg: Driscoll-Kraay SEs
res_dk = pr.ols(
    "lgsp ~ lpcap + lpc + lemp + unemp",
    data=df_produc,
    vcov="DK",
    time="year",
)
res_dk.summary()

In [ ]:
# R: fixest with Driscoll-Kraay
r_dk = run_r_regression(f'''
library(fixest)
{R_PRODUC}

model <- feols(lgsp ~ lpcap + lpc + lemp + unemp, data=df,
               panel.id = ~state + year, vcov = "DK")
vcov_mat <- vcov(model)
{R_EXTRACT}
''')
print("R coefficients:", r_dk.coef)
print("R SEs:", r_dk.se)

In [ ]:
print("=== Driscoll-Kraay SEs ===")
compare(res_dk, r_dk, rtol=5e-3, label="DK")

<cell_type>markdown</cell_type>---
### Test 3: Newey-West with Panel FE

`panel_fe(entity="state", time="year")` demeans by both entity and time FE, then
applies NW on the demeaned residuals.

R comparison uses `fixest::feols()` with two-way FE (`| state + year`) and `vcov="NW"`.

In [ ]:
# polars_reg: Panel FE with Newey-West SEs
res_fe_nw = pr.panel_fe(
    "lgsp ~ lpcap + lpc + lemp + unemp",
    data=df_produc,
    entity="state",
    time="year",
    vcov="NW",
)
res_fe_nw.summary()

In [ ]:
# R: fixest with entity + year FE and Newey-West
# panel_fe(entity="state", time="year") absorbs BOTH state and year FE,
# so R must also include year in the FE formula to match.
r_fe_nw = run_r_regression(f'''
library(fixest)
{R_PRODUC}

model <- feols(lgsp ~ lpcap + lpc + lemp + unemp | state + year, data=df,
               panel.id = ~state + year, vcov = "NW")
vcov_mat <- vcov(model)
{R_EXTRACT}
''')
print("R coefficients:", r_fe_nw.coef)
print("R SEs:", r_fe_nw.se)

In [ ]:
print("=== Panel FE + Newey-West ===")
compare(res_fe_nw, r_fe_nw, rtol=5e-3, se_rtol=0.5, label="Panel FE + NW")

---
## Weighted Least Squares

Using `mtcars` dataset with model `mpg ~ wt + hp` and weights `w = 1/wt`.

### Test 4: WLS with iid standard errors

Should match R's `lm(..., weights=w)` to machine precision.

In [ ]:
# polars_reg: WLS with iid SEs
res_wls = pr.ols(
    "mpg ~ wt + hp",
    data=df_mtcars,
    weights="w",
)
res_wls.summary()

In [ ]:
# R: lm() with weights, iid vcov
r_wls = run_r_regression(f'''
{R_MTCARS}

model <- lm(mpg ~ wt + hp, data=df, weights=w)
vcov_mat <- vcov(model)
{R_EXTRACT}
''')
print("R coefficients:", r_wls.coef)
print("R SEs:", r_wls.se)

In [ ]:
print("=== WLS iid ===")
compare(res_wls, r_wls, rtol=1e-8, label="WLS iid")

---
### Test 5: WLS with robust (HC1) standard errors

R comparison uses `sandwich::vcovHC(model, type="HC1")`.

In [ ]:
# polars_reg: WLS with HC1 robust SEs
res_wls_hc1 = pr.ols(
    "mpg ~ wt + hp",
    data=df_mtcars,
    weights="w",
    vcov="HC1",
)
res_wls_hc1.summary()

In [ ]:
# R: lm() with weights, HC1 robust vcov
r_wls_hc1 = run_r_regression(f'''
library(sandwich)
library(lmtest)
{R_MTCARS}

model <- lm(mpg ~ wt + hp, data=df, weights=w)
vcov_mat <- vcovHC(model, type="HC1")
{R_EXTRACT}
''')
print("R coefficients:", r_wls_hc1.coef)
print("R SEs:", r_wls_hc1.se)

In [ ]:
print("=== WLS HC1 ===")
compare(res_wls_hc1, r_wls_hc1, rtol=1e-5, label="WLS HC1")

<cell_type>markdown</cell_type>---
## Summary

| # | Test | polars_reg | R reference | Tolerance |
|---|------|-----------|-------------|----------|
| 1 | Newey-West SEs | `ols(vcov="NW", time=)` | `sandwich::NeweyWest()` | 5e-3 |
| 2 | Driscoll-Kraay SEs | `ols(vcov="DK", time=)` | `fixest::feols(vcov="DK")` | 5e-3 |
| 3 | Panel FE + NW | `panel_fe(vcov="NW")` | `fixest::feols(... \| state + year, vcov="NW")` | 5e-3 |
| 4 | WLS iid | `ols(weights=)` | `lm(weights=)` | 1e-8 |
| 5 | WLS HC1 | `ols(weights=, vcov="HC1")` | `sandwich::vcovHC(type="HC1")` | 1e-5 |

Run all cells to fill in PASS/FAIL results above.

In [ ]:
# Cleanup temp CSVs
for f in [csv_produc, csv_mtcars]:
    if os.path.exists(f):
        os.remove(f)
print("Cleaned up temp CSVs.")